# 4-3절 연습 문제 풀이

이 노트북은 4-3절 연습 문제(4-6 ~ 4-9)의 풀이 예시다.

- 본문 예제 코드는 `code_examples/ch04/04-03_example.ipynb`를 참고한다.
- 내려받는 데이터셋은 저장소 규약(공통코드컨벤션 13.9)에 따라 `download/` 디렉터리에 저장한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다.

In [1]:
import copy
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DOWNLOAD_ROOT = '../../download'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'학습 장치: {device}')

학습 장치: cuda


## 연습 문제 4-6

> CIFAR-10 데이터셋은 MNIST만큼이나 딥러닝 교육에 많이 사용되는 유명한 데이터셋으로,
> `torchvision.datasets`의 `CIFAR10` 클래스를 통해 파이토치 내장 데이터셋으로 제공된다.
> MNIST 데이터셋처럼 10개의 클래스로 구성되어 있는데, 이미지 샘플이 32x32 크기의 3채널 컬러 이미지(사진)라는 점이 다르다.
> - CIFAR-10 데이터셋에서 하나의 이미지 샘플은 몇 개의 숫자로 표현될까?
> - CIFAR-10 데이터셋으로 다층 퍼셉트론 모델을 학습한다고 가정하고 데이터셋과 데이터로더를 만들어 보자
>   (표준화 기준 평균과 표준편차는 모든 채널에 0.5를 사용한다).

### 손으로 푼 풀이 — 첫 번째 물음

본문 p27의 이미지 텐서 표기를 그대로 적용하면 된다. 32x32 크기의 3채널 컬러 이미지이므로
텐서 형태는 `(C, H, W)` = `(3, 32, 32)`이고, 숫자의 개수는 **3 × 32 × 32 = 3,072개**다.

MNIST와 비교하면 차이가 분명해진다. MNIST는 1채널 28x28이라 784개인데, CIFAR-10은 그 **네 배 가까이** 된다.
다층 퍼셉트론에 넣으려면 평탄화해서 `(3072,)` 형태로 만들어야 하고, 그러면 첫 은닉층의 가중치만 해도
3,072 × 뉴런 수가 된다. 이미지가 조금만 커져도 완전 연결 계층의 파라미터가 폭발적으로 늘어난다는 뜻인데,
이것이 5장에서 합성곱 신경망이 등장하는 이유이기도 하다.

In [2]:
BATCH_SIZE = 100

# MNIST 예제와 같은 구성이되, 채널이 3개이므로 평균과 표준편차도 채널 수만큼 지정한다
transform_chain = transforms.Compose([
    transforms.ToTensor(),                                  # (H, W, 3) -> (3, 32, 32), 0~1로 정규화
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # 채널마다 평균 0.5, 표준편차 0.5
    nn.Flatten(start_dim=0),                                # (3, 32, 32) -> (3072,)
])

train_valid_set = datasets.CIFAR10(root=DOWNLOAD_ROOT, train=True, download=True,
                                   transform=transform_chain)
test_set = datasets.CIFAR10(root=DOWNLOAD_ROOT, train=False, download=True,
                            transform=transform_chain)

# 훈련 데이터셋 50,000개를 40,000개와 10,000개로 나눠 훈련과 검증에 사용
generator = torch.Generator().manual_seed(SEED)
train_set, valid_set = random_split(train_valid_set, [40000, 10000], generator=generator)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

inputs, labels = next(iter(train_loader))
print(f'훈련 {len(train_set)}개, 검증 {len(valid_set)}개, 평가 {len(test_set)}개')
print(f'배치 입력 텐서 {tuple(inputs.shape)}, 배치 정답 텐서 {tuple(labels.shape)}')
print(f'샘플 하나의 숫자 개수: {inputs[0].numel()}개')
print(f'클래스: {train_valid_set.classes}')

훈련 40000개, 검증 10000개, 평가 10000개
배치 입력 텐서 (100, 3072), 배치 정답 텐서 (100,)
샘플 하나의 숫자 개수: 3072개
클래스: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


### 풀이 해설

MNIST 예제([코드 4-19])에서 바뀌는 곳은 **두 군데뿐이다.**

1. `datasets.MNIST` → `datasets.CIFAR10`
2. `transforms.Normalize`의 평균과 표준편차를 **채널 수만큼** 지정한다. MNIST는 1채널이라 `(0.5,)`,
   CIFAR-10은 3채널이라 `(0.5, 0.5, 0.5)`다. 여기서 틀리면 예외가 발생한다.

`ToTensor`가 `(H, W, 3)` 넘파이 배열을 `(3, 32, 32)` 텐서로 바꾸고 픽셀값을 0~1로 정규화하며,
`Normalize`가 평균 0.5, 표준편차 0.5로 표준화해 -1~1 범위로 옮긴다. 마지막 `nn.Flatten`이 `(3072,)`로 편다.
본문 각주 17이 설명한 흐름 그대로다.

실행 결과에서 배치 입력 텐서가 `(100, 3072)`임을 확인할 수 있다.

### 문제 검토

- **적절성: 적합.** 본문에서 배운 데이터 파이프라인을 다른 데이터셋에 그대로 옮겨 보게 한다.
  MNIST와 CIFAR-10의 차이가 채널 수와 크기뿐이라 바뀌는 곳이 적어, 파이프라인이 데이터에 독립적이라는 점이 드러난다.
  첫 번째 물음(숫자 개수)이 5장의 합성곱 신경망 도입과도 이어져 배치가 좋다.
- **[검토] 채널 수만큼 평균·표준편차를 지정해야 한다는 점이 숨은 관문이다.** 지문은 "모든 채널에 0.5를 사용한다"고
  하지만, MNIST 코드를 그대로 복사해 `(0.5,)`로 두면 예외가 발생한다. 이 문제에서 독자가 실제로 막히는 유일한 지점이므로
  힌트로 남기거나, 반대로 그대로 두어 스스로 발견하게 할지 판단이 필요하다.
- **[검토] 다층 퍼셉트론에 넣으려면 평탄화가 필요하다.** 지문은 데이터셋과 데이터로더까지만 요구하는데,
  "다층 퍼셉트론 모델을 학습한다고 가정하고"라고 했으므로 평탄화까지 포함하는 것이 자연스럽다.
  본문 [코드 4-18]의 `transform_chain`에 `nn.Flatten`이 들어 있으므로 독자가 유추할 수는 있다.

## 연습 문제 4-7

> 256개와 128개의 뉴런을 가진 두 개의 은닉층을 사용한 이번 절의 다층 퍼셉트론 모델의 구조를 바꿔 가며 실험해,
> 검증 손실과 평가 정확도가 더 좋은 모델을 만들어 보자. 실험 과정에서 중요한 재현 가능성도 염두에 두고 진행하자.

In [3]:
# MNIST 데이터 준비 (본문 코드 4-19와 같은 구성)
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    nn.Flatten(start_dim=0),
])
mnist_train_valid = datasets.MNIST(root=DOWNLOAD_ROOT, train=True, download=True,
                                   transform=mnist_transform)
mnist_test = datasets.MNIST(root=DOWNLOAD_ROOT, train=False, download=True,
                            transform=mnist_transform)
generator = torch.Generator().manual_seed(SEED)
mnist_train, mnist_valid = random_split(mnist_train_valid, [50000, 10000], generator=generator)

m_train_loader = DataLoader(mnist_train, batch_size=100, shuffle=True)
m_valid_loader = DataLoader(mnist_valid, batch_size=100, shuffle=False)
m_test_loader = DataLoader(mnist_test, batch_size=100, shuffle=False)
print(f'MNIST 훈련 {len(mnist_train)}개, 검증 {len(mnist_valid)}개, 평가 {len(mnist_test)}개')

MNIST 훈련 50000개, 검증 10000개, 평가 10000개


In [4]:
EPOCHS = 20
PATIENCE = 3
LR = 0.001

def build_classifier(hidden_sizes, in_features=784, out_features=10):
    layers, prev = [], in_features
    for size in hidden_sizes:
        layers += [nn.Linear(prev, size), nn.ReLU()]
        prev = size
    layers.append(nn.Linear(prev, out_features))
    return nn.Sequential(*layers)

def run_epoch(model, loader, criterion, optimizer=None):
    """optimizer가 있으면 학습, 없으면 평가. 가중 평균 손실과 정확도를 돌려준다."""
    model.train() if optimizer else model.eval()
    total_loss, correct, total = 0., 0, 0
    with torch.set_grad_enabled(optimizer is not None):
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            if optimizer:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (outputs.argmax(dim=-1) == labels).sum().item()
            total += len(labels)
    return total_loss / total, correct / total * 100

def train_model(hidden_sizes, seed=SEED):
    torch.manual_seed(seed)          # 재현 가능성: 구조마다 같은 시드로 초기화
    model = build_classifier(hidden_sizes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    best_loss, best_epoch, best_params, counter = float('inf'), 0, None, 0
    for epoch in range(1, EPOCHS + 1):
        run_epoch(model, m_train_loader, criterion, optimizer)
        valid_loss, _ = run_epoch(model, m_valid_loader, criterion)
        if valid_loss < best_loss:
            best_loss, best_epoch, counter = valid_loss, epoch, 0
            best_params = copy.deepcopy(model.state_dict())
        else:
            counter += 1
            if counter >= PATIENCE:
                break
    model.load_state_dict(best_params)
    _, test_acc = run_epoch(model, m_test_loader, criterion)
    n_params = sum(p.numel() for p in model.parameters())
    return best_epoch, best_loss, test_acc, n_params

In [5]:
STRUCTURES = {
    '256, 128 (본문)': [256, 128],
    '128': [128],
    '512': [512],
    '512, 256': [512, 256],
    '256, 128, 64': [256, 128, 64],
    '1024, 512, 256': [1024, 512, 256],
}

print(f'{"구조":>18} {"파라미터":>10} {"최적 에포크":>10} {"검증 손실":>10} {"평가 정확도":>11}')
print('-' * 66)
results = {}
for name, hidden_sizes in STRUCTURES.items():
    best_epoch, best_loss, test_acc, n_params = train_model(hidden_sizes)
    results[name] = (best_loss, test_acc)
    print(f'{name:>18} {n_params:10,d} {best_epoch:10d} {best_loss:10.4f} {test_acc:10.2f}%')

                구조       파라미터     최적 에포크      검증 손실      평가 정확도
------------------------------------------------------------------


     256, 128 (본문)    235,146          7     0.0821      97.47%


               128    101,770          9     0.0993      97.24%


               512    407,050          9     0.0806      97.59%


          512, 256    535,818          8     0.0785      97.53%


      256, 128, 64    242,762         10     0.0905      97.24%


    1024, 512, 256  1,462,538         10     0.0787      98.06%


### 풀이 해설

여섯 가지 구조를 같은 조건(시드, 학습률, 조기 종료 설정)으로 학습해 비교했다.

지문이 강조한 **재현 가능성**을 위해 세 가지를 맞췄다.

1. 구조마다 `torch.manual_seed(SEED)`를 **다시 호출**해 같은 시드에서 파라미터를 초기화한다.
   그러지 않으면 앞 실험이 소비한 난수 때문에 뒤 실험의 초깃값이 달라져 비교가 공정하지 않다.
2. 데이터 분할에도 시드를 고정한 `generator`를 넘겨 훈련·검증 구성이 매번 같게 했다(본문 p20).
3. 학습률, 에포크, 참을성 한계 등 나머지 조건을 모두 동일하게 두어 **구조만 변수가 되도록** 했다.

결과를 보면 은닉층을 넓히거나 깊게 하면 파라미터가 크게 늘지만 정확도는 그만큼 오르지 않는다.
MNIST는 다층 퍼셉트론으로 이미 97~98%에 이르는 비교적 쉬운 문제라, 구조를 키워 얻을 수 있는 여지가 크지 않다.
여기서 더 올리려면 구조가 아니라 **모델 종류**를 바꿔야 하는데, 그것이 5장의 합성곱 신경망이다.

### 문제 검토

- **적절성: 적합.** 하이퍼파라미터 탐색을 직접 해 보게 하는 실전형 문제다. 특히 "재현 가능성도 염두에 두고"라는
  한 구절이 이 문제를 살린다. 여러 구조를 순서대로 학습하면 난수 상태가 이어져 비교가 공정하지 않다는 점을
  스스로 깨닫게 하는 장치다.
- **[검토] 탐색 범위가 열려 있다.** '구조를 바꿔 가며'만으로는 은닉층 수, 뉴런 수, 활성화 함수 중 무엇을
  얼마나 바꿔야 할지 알 수 없다. 서너 가지 후보를 예시로 들어 주면 독자가 방향을 잡기 쉽다.
- **[검토] '더 좋은 모델'의 기준이 두 개다.** 검증 손실과 평가 정확도를 함께 보라고 하는데, 둘이 어긋날 때
  무엇을 따를지가 정해져 있지 않다. 연습 문제 4-4와 이어지는 질문이므로, "둘이 어긋난다면 어느 쪽을 믿어야
  할지도 생각해 보자"를 덧붙이면 좋다.
- **[검토] 재현 가능성의 구체적 방법을 짚어 주면 좋다.** 시드 고정이 필요한 지점이 모델 초기화와 데이터 분할
  두 곳인데, 지문만으로는 어디에 무엇을 해야 할지 알기 어렵다.

## 연습 문제 4-8

> 숫자 분류기 모델의 분류 결과로부터 10x10 형태의 혼동 행렬 텐서를 만드는 함수를 작성하고, 혼동 행렬을 출력해 보자.
> 참고로 많은 데이터 분석 라이브러리에 혼동 행렬 계산 기능이 포함되어 있다.
> 하지만, 여기서는 텐서와 관련된 파이토치 함수와 메서드만 사용해 도전해 보길 권한다.

In [6]:
NUM_CLASSES = 10

def confusion_matrix(model, loader, num_classes=NUM_CLASSES):
    """행은 실제 클래스, 열은 예측 클래스인 혼동 행렬을 만든다."""
    matrix = torch.zeros(num_classes, num_classes, dtype=torch.int64)
    model.eval()
    with torch.no_grad():
        for inputs, labels in loader:
            preds = model(inputs.to(device)).argmax(dim=-1).cpu()
            # (실제, 예측) 쌍을 하나의 번호로 합친 뒤 bincount로 한 번에 집계
            pair_index = labels * num_classes + preds
            matrix += torch.bincount(pair_index, minlength=num_classes ** 2).reshape(num_classes, num_classes)
    return matrix

# 본문과 같은 구조(256, 128)의 모델을 학습해 혼동 행렬을 만든다
torch.manual_seed(SEED)
model = build_classifier([256, 128]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
for epoch in range(5):
    run_epoch(model, m_train_loader, criterion, optimizer)
_, test_acc = run_epoch(model, m_test_loader, criterion)
print(f'평가 정확도: {test_acc:.2f}%\n')

matrix = confusion_matrix(model, m_test_loader)
print('      ' + ''.join(f'{i:6d}' for i in range(NUM_CLASSES)) + '   (열: 예측)')
for i, row in enumerate(matrix.tolist()):
    print(f'실제 {i} ' + ''.join(f'{v:6d}' for v in row))
print(f'\n대각선 합(맞힌 샘플): {matrix.diag().sum().item()} / {matrix.sum().item()}')

평가 정확도: 96.61%



           0     1     2     3     4     5     6     7     8     9   (열: 예측)
실제 0    972     1     1     0     1     0     2     1     1     1
실제 1      0  1122     2     4     0     1     2     0     3     1
실제 2      5     0  1008     5     2     0     3     7     2     0
실제 3      2     0     3   991     0     0     0     9     4     1
실제 4      3     3     1     0   942     0     8     7     2    16
실제 5     12     1     2    33     2   818     6     2     8     8
실제 6      8     3     2     2     3     5   932     0     3     0
실제 7      2     6     9     2     0     0     0   997     0    12
실제 8      8     0     9    26     3     1     4     6   914     3
실제 9      6     4     2    11     8     0     2    10     1   965

대각선 합(맞힌 샘플): 9661 / 10000


In [7]:
# 가장 자주 혼동한 쌍을 찾아본다
errors = matrix.clone()
errors.fill_diagonal_(0)                       # 맞힌 것(대각선)은 지운다
top_values, flat_indices = errors.flatten().topk(5)
print('자주 틀린 조합 다섯 가지')
for value, index in zip(top_values.tolist(), flat_indices.tolist()):
    actual, predicted = index // NUM_CLASSES, index % NUM_CLASSES
    print(f'  실제 {actual} -> 예측 {predicted}: {value}회')

# 클래스별 재현율(본문 p34)
recall = matrix.diag().float() / matrix.sum(dim=1).float() * 100
print('\n클래스별 재현율')
print('  ' + '  '.join(f'{i}:{r:.1f}%' for i, r in enumerate(recall.tolist())))

자주 틀린 조합 다섯 가지
  실제 5 -> 예측 3: 33회
  실제 8 -> 예측 3: 26회
  실제 4 -> 예측 9: 16회
  실제 5 -> 예측 0: 12회
  실제 7 -> 예측 9: 12회

클래스별 재현율
  0:99.2%  1:98.9%  2:97.7%  3:98.1%  4:95.9%  5:91.7%  6:97.3%  7:97.0%  8:93.8%  9:95.6%


### 풀이 해설

혼동 행렬은 `행 = 실제 클래스, 열 = 예측 클래스`인 10x10 표다. `matrix[i][j]`는 실제로 i인데 j로 예측한 샘플 수다.
대각선은 맞힌 샘플, 나머지는 틀린 샘플이다.

파이토치 함수만으로 만드는 요령은 **(실제, 예측) 쌍을 하나의 번호로 합치는 것**이다.
`실제 * 10 + 예측`으로 계산하면 0~99의 번호가 되고, `torch.bincount()`로 각 번호의 개수를 한 번에 세어
`reshape(10, 10)`으로 펴면 혼동 행렬이 된다. 이중 루프 없이 배치마다 한 줄로 집계할 수 있다.

결과를 보면 본문 p34가 말한 "자주 발생하는 분류 실패의 경향"이 드러난다.
정확도라는 숫자 하나로는 보이지 않던 것이 표에서는 보인다. 예를 들어 서로 닮은 숫자끼리 혼동이 몰리는 경향이 있는데,
이는 모델이 무작위로 틀리는 것이 아니라 **특정 형태를 구분하지 못한다**는 뜻이다.

행 방향으로 합을 내면 각 클래스의 실제 샘플 수가 되므로, 대각선을 그 합으로 나누면 본문 p34의 **재현율**이 된다.
열 방향으로 나누면 **정밀도**가 된다. 혼동 행렬 하나에서 세 지표가 모두 나온다.

### 문제 검토

- **적절성: 적합.** 본문이 "혼동 행렬을 시각화하는 예제는 깃허브 예제 노트북에 게시해 두었다"며 넘긴 내용을
  독자가 직접 만들어 보게 한다. "파이토치 함수와 메서드만 사용해"라는 제약이 특히 좋다.
  그 제약 덕분에 1장에서 배운 인덱싱과 집계를 다시 꺼내 쓰게 된다.
- **[검토] 만든 다음에 무엇을 볼지 없다.** '출력해 보자'로 끝나 10x10 숫자 표를 보고 덮을 수 있다.
  혼동 행렬의 가치는 **어떤 쌍을 자주 혼동하는지 찾는 데** 있으므로 한 구절 덧붙이면 좋다.
- **[검토] 본문의 세 지표와 연결하면 좋다.** 바로 앞 p34에서 정밀도, 재현율, F1 점수를 소개하는데,
  이 세 지표가 모두 혼동 행렬에서 계산된다는 점을 묻지 않는다. 좋은 연결 고리를 놓치고 있다.

**윤문안**

> **4-8**. 숫자 분류기 모델의 분류 결과로부터 10x10 형태의 혼동 행렬 텐서를 만드는 함수를 작성하고,
> 혼동 행렬을 출력해 보자. 그리고 모델이 가장 자주 혼동하는 숫자 쌍을 찾고, 혼동 행렬에서 클래스별 재현율을
> 계산해 보자. 참고로 많은 데이터 분석 라이브러리에 혼동 행렬 계산 기능이 포함되어 있다.
> 하지만, 여기서는 텐서와 관련된 파이토치 함수와 메서드만 사용해 도전해 보길 권한다.

## 연습 문제 4-9 [도전 문제]

> 순서대로 계이름을 나열하면 음악도 분석 가능한 데이터가 된다. 다음은 동요 <반짝반짝 작은별>의 계이름(쉼표 포함)으로,
> 도레미파솔라시를 각각 0123456으로, 쉼표를 7로 표현한 것이다.
>
> `004455473322110744332217443322170044554733221107`
>
> 이 데이터를 학습해서 8개의 계이름(쉼표 포함)을 입력하면 다음 음 또는 쉼표를 예측하는 다층 퍼셉트론 모델을
> 만들어 보자. 순서가 있는 데이터(순차 데이터)를 처리하는 모델은 6장에서 소개하지만, 결과가 좋지 않더라도
> 상관없으니 일단 4장까지 학습한 내용을 총동원해 도전해 보자.

In [8]:
MELODY = '004455473322110744332217443322170044554733221107'
WINDOW = 8          # 입력으로 사용할 계이름의 수
NUM_NOTES = 8       # 0~6은 도레미파솔라시, 7은 쉼표

notes = torch.tensor([int(ch) for ch in MELODY])
print(f'전체 길이 {len(notes)}개, 사용하는 기호 {sorted(set(notes.tolist()))}')

# 8개를 보고 다음 1개를 맞히는 샘플을 만든다 (슬라이딩 윈도)
X_list, Y_list = [], []
for i in range(len(notes) - WINDOW):
    X_list.append(notes[i:i + WINDOW])
    Y_list.append(notes[i + WINDOW])
X_seq = torch.stack(X_list)
Y_seq = torch.stack(Y_list)
print(f'샘플 {len(X_seq)}개, 입력 {tuple(X_seq.shape)}, 정답 {tuple(Y_seq.shape)}')
print(f'첫 샘플: 입력 {X_seq[0].tolist()} -> 정답 {Y_seq[0].item()}')

전체 길이 48개, 사용하는 기호 [0, 1, 2, 3, 4, 5, 7]
샘플 40개, 입력 (40, 8), 정답 (40,)
첫 샘플: 입력 [0, 0, 4, 4, 5, 5, 4, 7] -> 정답 3


In [9]:
import torch.nn.functional as F

# 계이름은 크기가 아니라 이름이므로 원-핫으로 바꿔 입력한다 (3장 p20)
X_onehot = F.one_hot(X_seq, num_classes=NUM_NOTES).float().flatten(start_dim=1)
print(f'원-핫 인코딩 후 입력 형태: {tuple(X_onehot.shape)}   (8개 × 8종류 = 64)')

def train_melody(X, Y, epochs=3000, hidden=64, seed=SEED):
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Linear(X.shape[1], hidden), nn.ReLU(),
        nn.Linear(hidden, hidden), nn.ReLU(),
        nn.Linear(hidden, NUM_NOTES),
    )
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = criterion(model(X), Y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        acc = (model(X).argmax(dim=-1) == Y).float().mean().item() * 100
    return model, loss.item(), acc

# 원-핫 입력과, 계이름 숫자를 그대로 넣은 경우를 비교
model_onehot, loss1, acc1 = train_melody(X_onehot, Y_seq)
model_raw, loss2, acc2 = train_melody(X_seq.float(), Y_seq)
print(f'원-핫 입력   : 손실 {loss1:.4f}, 학습 데이터 정확도 {acc1:.2f}%')
print(f'숫자 그대로 입력: 손실 {loss2:.4f}, 학습 데이터 정확도 {acc2:.2f}%')

원-핫 인코딩 후 입력 형태: (40, 64)   (8개 × 8종류 = 64)


원-핫 입력   : 손실 0.0347, 학습 데이터 정확도 97.50%
숫자 그대로 입력: 손실 0.0347, 학습 데이터 정확도 97.50%


In [10]:
# 학습한 모델로 멜로디를 이어서 생성해 본다
NOTE_NAMES = ['도', '레', '미', '파', '솔', '라', '시', '(쉼)']

def generate(model, seed_notes, length=16):
    current = list(seed_notes)
    for _ in range(length):
        window = torch.tensor(current[-WINDOW:])
        x = F.one_hot(window, num_classes=NUM_NOTES).float().flatten().unsqueeze(0)
        with torch.no_grad():
            current.append(model(x).argmax(dim=-1).item())
    return current

seed_notes = notes[:WINDOW].tolist()
generated = generate(model_onehot, seed_notes)
print(f'입력한 앞 8개 : {" ".join(NOTE_NAMES[n] for n in seed_notes)}')
print(f'이어서 생성   : {" ".join(NOTE_NAMES[n] for n in generated[WINDOW:])}')
print(f'실제 원곡     : {" ".join(NOTE_NAMES[n] for n in notes[WINDOW:WINDOW + 16].tolist())}')

입력한 앞 8개 : 도 도 솔 솔 라 라 솔 (쉼)
이어서 생성   : 파 파 미 미 레 레 도 (쉼) 솔 솔 파 파 미 미 레 (쉼)
실제 원곡     : 파 파 미 미 레 레 도 (쉼) 솔 솔 파 파 미 미 레 (쉼)


### 풀이 해설

4장까지 배운 내용만으로 순차 데이터를 다루는 방법은 **슬라이딩 윈도**다.
연속한 8개를 입력, 그다음 1개를 정답으로 삼아 샘플을 만들면 48개 길이의 멜로디에서 40개의 샘플이 나온다.

입력은 3장의 교훈에 따라 원-핫으로 인코딩했다. **계이름은 크기가 아니라 이름이다.**
'미'(2)가 '도'(0)의 두 배라는 뜻이 아니므로, 숫자를 그대로 넣으면 모델이 잘못된 크기 관계를 학습할 수 있다.
3장 p20이 정답 레이블에 대해 설명한 원-핫 인코딩을 **입력에도** 적용하는 것이 원칙이다.
입력 크기는 8개 × 8종류 = 64가 된다.

**다만 이 데이터에서는 두 방식의 차이가 드러나지 않는다.** 위 실행 결과에서 원-핫 입력과 숫자 그대로 입력이
손실 0.0347, 정확도 97.50%로 완전히 같다. 에포크를 100으로 줄여도 마찬가지다.
샘플이 40개뿐이고 값의 종류도 8가지뿐이라, 모델이 두 경우 모두 **데이터를 통째로 외워 버리기** 때문이다.
원-핫 인코딩의 필요성은 원칙으로는 분명하지만, 이 문제의 데이터로는 실증되지 않는다는 점을 함께 기억해 두자.

학습한 모델로 멜로디를 이어 생성해 보면 원곡을 그대로 따라간다. 이 역시 **데이터를 외운 결과**다.
같은 패턴이 반복되는 동요를 40개 샘플로 학습했으므로, 모델이 일반적인 음악 규칙을 배웠다고 보기는 어렵다.

그리고 이 방식에는 구조적 한계가 있다. 입력 길이가 8로 고정되어 **더 긴 맥락을 볼 수 없고**,
8개 안에서도 순서 정보가 위치별 가중치로만 표현되어 **'앞에서 본 것을 기억한다'는 개념이 없다.**
지문이 예고한 대로 6장의 순환 신경망이 바로 이 두 가지를 해결한다.

### 문제 검토

- **적절성: 적합. 1부를 닫는 도전 문제로 훌륭하다.** "결과가 좋지 않더라도 상관없으니 일단 도전해 보자"라는
  지문이 이 문제의 성격을 정확히 설명한다. 다층 퍼셉트론으로 순차 데이터를 다뤄 보고 그 한계를 몸으로 느낀 뒤
  6장으로 넘어가게 하는 구성이다. 데이터도 48글자짜리 문자열 하나뿐이라 부담이 없다.
- **[중요] 데이터가 너무 작아 원-핫 인코딩의 필요성이 드러나지 않는다.** 이 문제의 핵심 학습 지점은
  '계이름은 크기가 없는 숫자이므로 원-핫으로 인코딩해야 한다'는 3장의 교훈을 입력에 적용하는 것이다.
  그런데 실제로 두 방식을 비교해 보면 손실도 정확도도 완전히 같다(0.0347, 97.50%). 샘플이 40개뿐이라
  모델이 두 경우 모두 외워 버리기 때문이다. 지문이 원-핫을 유도하더라도 독자가 '왜 굳이?'라고 느낄 수 있다.
  멜로디를 더 늘리거나(연습 문제 6-8처럼 동요 네 곡), 검증용 샘플을 따로 두면 차이가 드러난다.
- **[검토] 무엇으로 결과를 확인할지 없다.** '다음 음을 예측하는 모델'까지만 요구하므로 정확도만 보고 끝낼 수 있다.
  학습한 모델로 멜로디를 이어 생성해 보면 결과가 눈과 귀에 들어와 훨씬 재미있다.
- **[검토] 8이라는 입력 길이의 의미.** 왜 하필 8개인지(한 마디), 길이를 바꾸면 어떻게 되는지를 물으면
  6장의 '맥락 길이' 개념으로 자연스럽게 이어진다.

**윤문안**

> **4-9**. [도전 문제] (앞부분 그대로) … 일단 4장까지 학습한 내용을 총동원해 도전해 보자.
> 학습한 모델로 앞 8개의 계이름에 이어지는 음을 차례로 예측해 멜로디를 만들어 보고, 원곡과 비교해 보자.
>
> 힌트: 계이름 0~7은 크기가 아니라 이름이다. 3장에서 정답 레이블을 다룰 때 사용한 방법을 입력에도 적용해 보자.